# Building Your First Neural Network

This notebook will guide you through the process of creating, training, and evaluating your first neural network model from scratch. We'll cover the fundamental concepts, implementation details, and best practices to help you understand how neural networks work.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand the basic components of neural networks
2. Build a simple neural network using TensorFlow/Keras
3. Train and evaluate the model's performance
4. Improve your model through various techniques
5. Save and load your trained model for future use

## 1. Import Required Libraries

Let's start by importing the necessary libraries for our neural network project:

In [ ]:
# Core libraries for data manipulation and analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras for building neural networks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D

# Scikit-learn for data preparation and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Understanding Neural Network Fundamentals

Neural networks are computational models inspired by the human brain. Here's a breakdown of their fundamental components:

### Key Components

1. **Neurons**: The basic units that receive input, apply a transformation, and produce output
2. **Layers**: Collections of neurons that process information:
   - **Input Layer**: Receives the raw data
   - **Hidden Layers**: Process the information from previous layers
   - **Output Layer**: Produces the final prediction

3. **Weights and Biases**: Parameters that determine the strength of connections between neurons and are adjusted during training

4. **Activation Functions**: Non-linear functions that determine the output of a neuron, introducing non-linearity into the model:
   - Sigmoid: Maps values to [0,1]
   - Tanh: Maps values to [-1,1]
   - ReLU (Rectified Linear Unit): Returns max(0,x)
   - Softmax: Converts values to probabilities (used in output layer for classification)

### Forward Propagation

The process of moving data through the network to generate predictions:
1. Input values are multiplied by weights
2. Results are summed together with a bias term
3. The sum passes through an activation function
4. The output serves as input to the next layer

### How Neural Networks Learn

During training, neural networks use:
1. **Loss Function**: Measures how far predictions are from actual values
2. **Backpropagation**: Calculates gradients of the loss function with respect to weights
3. **Optimization**: Updates weights to minimize the loss (e.g., Gradient Descent)

In [ ]:
# Visual representation of a simple neural network
plt.figure(figsize=(12, 8))

# Layers
layer_sizes = [4, 5, 5, 3]  # Input, hidden, hidden, output
layer_positions = [1, 2.5, 4, 5.5]
layer_names = ['Input\nLayer', 'Hidden\nLayer 1', 'Hidden\nLayer 2', 'Output\nLayer']
colors = ['#3498db', '#2ecc71', '#2ecc71', '#e74c3c']

# Draw nodes
for i, (n_nodes, pos, name, color) in enumerate(zip(layer_sizes, layer_positions, layer_names, colors)):
    # Plot nodes
    y_positions = np.linspace(0, n_nodes-1, n_nodes) - (n_nodes-1)/2
    for y in y_positions:
        plt.scatter(pos, y, s=700, color=color, edgecolor='black', zorder=3)
    
    # Layer name
    plt.text(pos, -3, name, ha='center', va='center', fontsize=14)
    
    # Connect to the next layer if it exists
    if i < len(layer_sizes)-1:
        next_y_positions = np.linspace(0, layer_sizes[i+1]-1, layer_sizes[i+1]) - (layer_sizes[i+1]-1)/2
        for y1 in y_positions:
            for y2 in next_y_positions:
                plt.plot([pos, layer_positions[i+1]], [y1, y2], 'k-', alpha=0.2, zorder=1)

plt.axis('off')
plt.title('Neural Network Architecture', fontsize=18)
plt.tight_layout()
plt.show()

### Common Activation Functions

Let's visualize some of the most common activation functions used in neural networks:

In [ ]:
# Plotting common activation functions
x = np.linspace(-5, 5, 200)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.1):
    return np.maximum(alpha * x, x)

plt.figure(figsize=(14, 8))

# Sigmoid
plt.subplot(2, 2, 1)
plt.plot(x, sigmoid(x), 'b-', linewidth=2)
plt.grid(True)
plt.title('Sigmoid')
plt.xlabel('x')
plt.ylabel('sigmoid(x)')
plt.ylim(-0.1, 1.1)

# Tanh
plt.subplot(2, 2, 2)
plt.plot(x, tanh(x), 'g-', linewidth=2)
plt.grid(True)
plt.title('Tanh')
plt.xlabel('x')
plt.ylabel('tanh(x)')
plt.ylim(-1.1, 1.1)

# ReLU
plt.subplot(2, 2, 3)
plt.plot(x, relu(x), 'r-', linewidth=2)
plt.grid(True)
plt.title('ReLU')
plt.xlabel('x')
plt.ylabel('ReLU(x)')
plt.ylim(-1, 5)

# Leaky ReLU
plt.subplot(2, 2, 4)
plt.plot(x, leaky_relu(x), 'm-', linewidth=2)
plt.grid(True)
plt.title('Leaky ReLU')
plt.xlabel('x')
plt.ylabel('Leaky ReLU(x)')
plt.ylim(-1, 5)

plt.tight_layout()
plt.show()

## 3. Preparing the Dataset

For this introductory neural network, we'll use the famous MNIST dataset of handwritten digits. This dataset contains 70,000 grayscale images of handwritten digits (0-9) with each image being 28x28 pixels.

### Loading the Dataset

In [ ]:
# Load the MNIST dataset from Keras
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Print dataset shape information
print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels shape: {y_test.shape}")

### Exploring the Dataset

Let's visualize some examples from the dataset to better understand what we're working with:

In [ ]:
# Visualize some examples from the dataset
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Check the distribution of classes
plt.figure(figsize=(10, 6))
plt.hist(y_train, bins=np.arange(11)-0.5, rwidth=0.8, alpha=0.7)
plt.xticks(range(10))
plt.title('Distribution of Digits in Training Set')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()

### Data Preprocessing

Before feeding the data into our neural network, we need to:
1. Normalize the pixel values to a range of [0, 1]
2. Reshape the data to fit the input shape expected by the neural network
3. Convert the target variable to one-hot encoded vectors

In [ ]:
# Normalize pixel values to range [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape data for the neural network
# For a simple feedforward neural network, we'll flatten the 28x28 images to 1D arrays
X_train_flat = X_train.reshape((X_train.shape[0], 28 * 28))
X_test_flat = X_test.reshape((X_test.shape[0], 28 * 28))

print(f"Flattened training data shape: {X_train_flat.shape}")
print(f"Flattened testing data shape: {X_test_flat.shape}")

# Convert class vectors to binary class matrices (one-hot encoding)
num_classes = 10
y_train_encoded = keras.utils.to_categorical(y_train, num_classes)
y_test_encoded = keras.utils.to_categorical(y_test, num_classes)

print(f"Original label for first sample: {y_train[0]}")
print(f"One-hot encoded label: {y_train_encoded[0]}")

## 4. Building a Simple Neural Network

Now, we'll build a simple feedforward neural network (also known as a multi-layer perceptron) using the Keras Sequential API. Our network will have:

1. An input layer (implicitly defined by the first hidden layer)
2. Two hidden layers with ReLU activation
3. An output layer with softmax activation for multi-class classification

In [ ]:
# Build a simple feedforward neural network
model = Sequential([
    # Input layer (implicitly defined) + first hidden layer
    Dense(128, activation='relu', input_shape=(28*28,)),
    
    # Second hidden layer
    Dense(64, activation='relu'),
    
    # Output layer with 10 neurons (one for each digit)
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display the model summary
model.summary()

### Understanding the Model Architecture

Let's break down our model:

1. **First Hidden Layer**:
   - 128 neurons with ReLU activation
   - Input shape: 784 (flattened 28x28 images)
   - Parameters: 784 * 128 (weights) + 128 (biases) = 100,480

2. **Second Hidden Layer**:
   - 64 neurons with ReLU activation
   - Parameters: 128 * 64 (weights) + 64 (biases) = 8,256

3. **Output Layer**:
   - 10 neurons with softmax activation (one for each digit)
   - Parameters: 64 * 10 (weights) + 10 (biases) = 650

The **ReLU activation** introduces non-linearity, allowing the network to learn complex patterns.

The **softmax activation** in the output layer converts the raw output values into probabilities that sum to 1, which is ideal for multi-class classification.

The **adam optimizer** is an adaptive learning rate optimization algorithm designed to handle sparse gradients.

The **categorical_crossentropy loss function** is appropriate for multi-class classification problems with mutually exclusive classes.

## 5. Training the Neural Network

Now we'll train our model using the preprocessed training data. During training:
1. The model makes predictions on batches of training data
2. The loss is calculated based on prediction errors
3. Gradients are computed through backpropagation
4. The optimizer updates the weights to minimize the loss
5. This process repeats for the specified number of epochs

In [ ]:
# Train the model
history = model.fit(
    X_train_flat,
    y_train_encoded,
    batch_size=128,
    epochs=10,
    validation_split=0.1,  # Use 10% of training data for validation
    verbose=1
)

## 6. Evaluating Model Performance

After training, it's important to evaluate how well the model performs on unseen data (our test set). This helps us understand if the model has learned meaningful patterns or just memorized the training data.

We'll look at several evaluation metrics:
1. Loss and accuracy on the test set
2. Learning curves from the training history
3. Confusion matrix to see which classes are confused with others

In [ ]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(X_test_flat, y_test_encoded)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
# Make predictions on test data
y_pred = model.predict(X_test_flat)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)

# Generate a classification report
print("Classification Report:\n")
print(classification_report(y_true, y_pred_classes))

# Create a confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 7. Visualizing the Learning Process

Let's visualize how our model's performance changed during training:

In [ ]:
# Plot training & validation accuracy and loss
plt.figure(figsize=(14, 5))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize some correct and incorrect predictions
def plot_prediction_examples(X, y_true, y_pred, n_examples=5):
    # Find correct and incorrect predictions
    correct_indices = np.where(y_true == y_pred)[0]
    incorrect_indices = np.where(y_true != y_pred)[0]
    
    # Get random samples
    np.random.shuffle(correct_indices)
    np.random.shuffle(incorrect_indices)
    
    # Plot examples
    plt.figure(figsize=(15, 6))
    
    # Plot correct predictions
    for i in range(n_examples):
        plt.subplot(2, n_examples, i+1)
        idx = correct_indices[i]
        plt.imshow(X[idx].reshape(28, 28), cmap='gray')
        plt.title(f"True: {y_true[idx]}\nPred: {y_pred[idx]}")
        plt.axis('off')
    
    # Plot incorrect predictions
    for i in range(n_examples):
        plt.subplot(2, n_examples, n_examples+i+1)
        idx = incorrect_indices[i]
        plt.imshow(X[idx].reshape(28, 28), cmap='gray')
        plt.title(f"True: {y_true[idx]}\nPred: {y_pred[idx]}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot examples
plot_prediction_examples(X_test_flat, y_true, y_pred_classes)

## 8. Making Predictions

Now that our model is trained, let's see how to use it to make predictions on new data. We'll select some test images and visualize the model's predictions including the confidence levels.

In [ ]:
# Select some random test examples
num_examples = 5
random_indices = np.random.choice(X_test.shape[0], num_examples, replace=False)
random_images = X_test_flat[random_indices]
random_labels = y_test[random_indices]

# Make predictions
predictions = model.predict(random_images)

# Visualize predictions
plt.figure(figsize=(15, 10))
for i in range(num_examples):
    plt.subplot(2, num_examples, i + 1)
    plt.imshow(X_test[random_indices[i]], cmap='gray')
    plt.title(f"True: {random_labels[i]}")
    plt.axis('off')
    
    # Plot prediction probabilities
    plt.subplot(2, num_examples, num_examples + i + 1)
    plt.bar(range(10), predictions[i])
    plt.title(f"Predicted: {np.argmax(predictions[i])}")
    plt.xticks(range(10))
    plt.ylim(0, 1)
    if i == 0:
        plt.ylabel('Probability')
    plt.xlabel('Digit')
    
plt.tight_layout()
plt.show()

## 9. Improving the Neural Network

Let's explore some techniques to improve our neural network's performance:

1. **Adding Dropout**: This helps prevent overfitting by randomly setting a fraction of input units to 0 during training
2. **Adding more neurons/layers**: This increases the model's capacity to learn complex patterns
3. **Batch Normalization**: This normalizes the activations of the previous layer at each batch, to decrease training time and potentially improve performance

In [ ]:
# Build an improved neural network
improved_model = Sequential([
    # Input layer (implicitly defined) + first hidden layer
    Dense(256, activation='relu', input_shape=(28*28,)),
    BatchNormalization(),
    Dropout(0.3),  # 30% dropout
    
    # Second hidden layer
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),  # 30% dropout
    
    # Third hidden layer
    Dense(64, activation='relu'),
    BatchNormalization(),
    
    # Output layer
    Dense(10, activation='softmax')
])

# Compile the model
improved_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display the model summary
improved_model.summary()

In [ ]:
# Train the improved model
improved_history = improved_model.fit(
    X_train_flat,
    y_train_encoded,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the improved model
improved_test_loss, improved_test_accuracy = improved_model.evaluate(X_test_flat, y_test_encoded)
print(f"Original model - Test accuracy: {test_accuracy:.4f}")
print(f"Improved model - Test accuracy: {improved_test_accuracy:.4f}")
print(f"Improvement: {(improved_test_accuracy - test_accuracy) * 100:.2f}%")

### Compare Model Performance

Let's compare the learning curves of both models to see the impact of our improvements:

In [ ]:
# Compare original and improved models
plt.figure(figsize=(15, 6))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Original Training')
plt.plot(history.history['val_accuracy'], label='Original Validation')
plt.plot(improved_history.history['accuracy'], label='Improved Training')
plt.plot(improved_history.history['val_accuracy'], label='Improved Validation')
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Original Training')
plt.plot(history.history['val_loss'], label='Original Validation')
plt.plot(improved_history.history['loss'], label='Improved Training')
plt.plot(improved_history.history['val_loss'], label='Improved Validation')
plt.title('Model Loss Comparison')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Saving and Loading the Model

Once you've trained a model that performs well, you'll want to save it for future use without having to retrain it each time. TensorFlow/Keras provides easy methods for saving and loading models.

In [ ]:
# Save the model architecture and weights
improved_model.save('c:/Users/pavel/projects/ai-ml/2-ML/mnist_model.h5')
print("Model saved successfully!")

# Save just the weights
improved_model.save_weights('c:/Users/pavel/projects/ai-ml/2-ML/mnist_model_weights.h5')
print("Model weights saved successfully!")

In [ ]:
# Load the complete model
loaded_model = load_model('c:/Users/pavel/projects/ai-ml/2-ML/mnist_model.h5')
print("Model loaded successfully!")

# Verify that the loaded model works correctly
loaded_model_loss, loaded_model_accuracy = loaded_model.evaluate(X_test_flat, y_test_encoded)
print(f"Loaded model - Test accuracy: {loaded_model_accuracy:.4f}")
print(f"Original improved model - Test accuracy: {improved_test_accuracy:.4f}")

## Summary and Next Steps

Congratulations! In this notebook, you've:

1. **Learned neural network fundamentals**: Neurons, layers, activation functions
2. **Prepared and explored data**: Loaded the MNIST dataset, normalized and reshaped it
3. **Built two neural networks**: A basic model and an improved version with regularization techniques
4. **Trained and evaluated models**: Used backpropagation and gradient descent to train the models
5. **Visualized performance**: Created plots to understand model behavior
6. **Made predictions**: Used the trained model for inference on new data
7. **Saved and loaded models**: Learned how to preserve your trained model

### Next Steps for Improvement:

1. **Try different architectures**: Experiment with the number of layers and neurons
2. **Use different optimizers**: Try RMSprop, SGD with momentum, or others
3. **Apply data augmentation**: Create variations of training images to improve generalization
4. **Try convolutional layers**: CNNs are better suited for image data
5. **Tune hyperparameters**: Learning rate, batch size, dropout rate
6. **Apply early stopping**: Prevent overfitting by stopping training when validation metrics stop improving

The concepts you've learned here form the foundation for building more complex neural networks for real-world problems.